In [ ]:
# Install required packages
import sys, subprocess
def pip_install(pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + pkgs)
try:
    import ultralytics
except Exception:
    pip_install(['ultralytics','pyyaml','pillow','matplotlib'])


# YOLOv8n on PASCAL VOC 2007 (local)
Notebook: conversion VOC XML -> YOLO labels + training with Ultralytics YOLOv8n.

In [ ]:
# Install Ultralytics if needed
import sys
import subprocess
def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])
try:
    import ultralytics
except Exception:
    pip_install('ultralytics')
    import ultralytics

In [ ]:
# Paths - adapt if needed
from pathlib import Path
DATA_ROOT = Path('Notebooks/099_YOLO/VOCtest_06-Nov-2007/VOCdevkit/VOC2007')
JPEG_DIR = DATA_ROOT / 'JPEGImages'
ANN_DIR = DATA_ROOT / 'Annotations'
IMAGESETS_MAIN = DATA_ROOT / 'ImageSets' / 'Main'
LABELS_DIR = DATA_ROOT / 'labels'
LABELS_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_ROOT ->', DATA_ROOT.resolve())

In [ ]:
# Pascal VOC classes (fixed order)
CLASSES = [
    'aeroplane','bicycle','bird','boat','bottle','bus','car','cat','chair','cow',
    'diningtable','dog','horse','motorbike','person','pottedplant','sheep','sofa','train','tvmonitor'
]
CLASS_MAP = {c: i for i, c in enumerate(CLASSES)}

In [ ]:
# Helper: parse VOC XML and write YOLO label files
import xml.etree.ElementTree as ET
def convert_xml_to_yolo(xml_path, img_w, img_h):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    yolo_lines = []
    for obj in root.findall('object'):
        name = obj.find('name').text
        if name not in CLASS_MAP:
            continue
        cls = CLASS_MAP[name]
        bnd = obj.find('bndbox')
        xmin = float(bnd.find('xmin').text)
        ymin = float(bnd.find('ymin').text)
        xmax = float(bnd.find('xmax').text)
        ymax = float(bnd.find('ymax').text)
        x_center = (xmin + xmax) / 2.0 / img_w
        y_center = (ymin + ymax) / 2.0 / img_h
        width = (xmax - xmin) / img_w
        height = (ymax - ymin) / img_h
        yolo_lines.append(f'{cls} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}')
    return yolo_lines

In [ ]:
# Build splits using ImageSets/Main or fall back to JSON lists
def read_split_file(path):
    if not path.exists():
        return []
    with open(path, 'r', encoding='utf-8') as f:
        names = [l.strip().split()[0] for l in f if l.strip()]
    return names
train_names = read_split_file(IMAGESETS_MAIN / 'train.txt')
val_names = read_split_file(IMAGESETS_MAIN / 'val.txt')
test_names = read_split_file(IMAGESETS_MAIN / 'test.txt')
# If some splits are empty, try trainval/test or use JSON files present in PASCAL_VOC folder
if not train_names and (IMAGESETS_MAIN / 'trainval.txt').exists():
    train_names = read_split_file(IMAGESETS_MAIN / 'trainval.txt')
print('Split sizes ->', len(train_names), len(val_names), len(test_names))

In [ ]:
# Convert all XML annotations to YOLO .txt labels for images referenced in splits
from pathlib import Path
count = 0
for xml in (ANN_DIR).glob('*.xml'):
    stem = xml.stem
    img_path = JPEG_DIR / f'{stem}.jpg'
    if not img_path.exists():
        continue
    # read image size from XML or use file system fallback
    tree = ET.parse(xml)
    root = tree.getroot()
    size = root.find('size')
    if size is not None:
        w = float(size.find('width').text)
        h = float(size.find('height').text)
    else:
        from PIL import Image
        w, h = Image.open(img_path).size
    yolo_lines = convert_xml_to_yolo(xml, w, h)
    if yolo_lines:
        out_path = LABELS_DIR / f'{stem}.txt'
        out_path.write_text('
'.join(yolo_lines), encoding='utf-8')
        count += 1
print('Wrote', count, 'label files to', LABELS_DIR)

In [ ]:
# Create train/val/test lists (absolute paths to images) and dataset.yaml
def abs_paths_for(names):
    return [str((JPEG_DIR / (n if n.endswith('.jpg') else n + '.jpg')).resolve()) for n in names]
train_list = abs_paths_for(train_names)
val_list = abs_paths_for(val_names)
test_list = abs_paths_for(test_names)
(DATA_ROOT / 'train.txt').write_text('
'.join(train_list), encoding='utf-8')
(DATA_ROOT / 'val.txt').write_text('
'.join(val_list), encoding='utf-8')
(DATA_ROOT / 'test.txt').write_text('
'.join(test_list), encoding='utf-8')
dataset_yaml = {
    'names': CLASSES,
    'nc': len(CLASSES),
    'train': str((DATA_ROOT / 'train.txt').resolve()),
    'val': str((DATA_ROOT / 'val.txt').resolve()),
    'test': str((DATA_ROOT / 'test.txt').resolve())
}
import yaml
with open(DATA_ROOT / 'dataset.yaml', 'w', encoding='utf-8') as f:
    yaml.safe_dump(dataset_yaml, f)
print('Wrote dataset.yaml to', (DATA_ROOT / 'dataset.yaml').resolve())

## Entraînement YOLOv8n

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')  # pretrained nano model
dataset_yaml = str((DATA_ROOT / 'dataset.yaml').resolve())
model.train(data=dataset_yaml, epochs=50, imgsz=640, batch=16, project='runs', name='yolov8n_voc2007')

In [ ]:
# Quick inference on a sample image and display results
from IPython.display import display
import matplotlib.pyplot as plt
sample = next((DATA_ROOT / 'JPEGImages').glob('*.jpg'))
res = model.predict(sample, save=False, imgsz=640)
res_plotted = res[0].plot()
plt.imshow(res_plotted); plt.axis('off')